# Prefect workflow for running the s3l0 eopf processor with the rs-dpr-service

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-652

See the associated:

  * Python module: [s3l0_demo_processor.py](./s3l0_demo_processor.py)
  * YAML file: [s3l0_demo_processor.yaml](./s3l0_demo_processor.yaml)

In [ ]:
import os

os.environ["RSPY_HOST_USER"] = "jgaucher" # change with yours
os.environ["RSPY_LOCAL_MODE"] = "1"
os.environ["RSPY_HOST_ADGS"] = "http://localhost:8001"
os.environ["RSPY_HOST_CADIP"] = "http://localhost:8002"
os.environ["RSPY_HOST_CATALOG"] = "http://localhost:8003"
os.environ["RSPY_HOST_STAGING"] = "http://localhost:8004"
os.environ["RSPY_DPR_SERVICE_ADDRESS"] = "http://localhost:6003"
os.environ["S3_ACCESSKEY"] = "minio"
os.environ["S3_SECRETKEY"] = "Strong#Pass#1234"
os.environ["S3_ENDPOINT"] = "http://localhost:9100"
os.environ["S3_REGION"] = "sbg"
os.environ["RSPY_TEMP_BUCKET"] = "rs-cluster-temp"
os.environ["RSPY_CATALOG_BUCKET"] = "rs-cluster-catalog"

os.environ["PREFECT_URL"] = os.environ["RSPY_PREFECT_URL"] = "http://localhost:4200"
os.environ["PREFECT_API_URL"] = os.environ["PREFECT_URL"] + "/api"
os.environ["PREFECT_WORK_POOL_STAGING"] = "pefect-pool-staging"
os.environ["PREFECT_WORK_POOL_EOPF"] = "pefect-pool-eopf"

os.environ["DASK_GATEWAY_STAGING_ADDRESS"] = os.environ["DASK_GATEWAY_STAGING_PUBLIC"] = "http://localhost:8701"
os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_PUBLIC"] = "http://localhost:8702"
os.environ["DASK_GATEWAY_EOPF_MOCKUP_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_PUBLIC"] = "http://localhost:8703"

os.environ["RSPY_OAUTH2_COOKIE"] = "dummy-cookie"


## 1. Initialisation

In [3]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://localhost:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [4]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *
USE_DPR_MOCKUP = True
if os.getenv("RSPY_LOCAL_MODE") == "1" and USE_DPR_MOCKUP:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_ADDRESS"]
    os.environ["DASK_GATEWAY_EOPF_PUBLIC"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_PUBLIC"]

init_demo()
init_dask_cluster_eopf(scale=2, use_mockup = USE_DPR_MOCKUP)
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)
display(dask_cluster_staging)

14:13:55.616 [WARNING] (rs_common.init_opentelemetry) 'TEMPO_ENDPOINT' variable is missing, cannot initialize OpenTelemetry


Auxip service: http://localhost:8001/auxip
CADIP service: http://localhost:8002/cadip
Catalog service: http://localhost:8003
Staging service: http://localhost:8004
Connecting to dask gateway for 'dask-eopf-mockup': http://localhost:8703 ...
image = cb3c543fbca34c169935e9976819cb8c
Get existing dask cluster: 'cb3c543fbca34c169935e9976819cb8c'
Dask dashboard for 'dask-eopf-mockup': http://localhost:8703/clusters/cb3c543fbca34c169935e9976819cb8c/status
Dask workers for 'dask-eopf-mockup' are up: 2/2
Connecting to dask gateway for 'dask-staging': http://localhost:8701 ...
image = 0db0f3c427ed460281430904d889b94d
Get existing dask cluster: '0db0f3c427ed460281430904d889b94d'
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/0db0f3c427ed460281430904d889b94d/status
Dask workers for 'dask-staging' are up: 2/2


/home/jgaucher/projects/rspy/working/demo-venv/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+----------+
| Package     | Client   | Scheduler | Workers  |
+-------------+----------+-----------+----------+
| dask        | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| distributed | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| lz4         | 4.4.3    | 4.4.4     | 4.4.4    |
| numpy       | 1.26.4   | 2.2.4     | 2.2.4    |
+-------------+----------+-----------+----------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/home/jgaucher/projects/rspy/working/demo-venv/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.3  | 4.4.4     | 4.4.4   |
| numpy   | 1.26.4 | 2.2.4     | 2.2.4   |
+

In [5]:
# Create a test collection
TEST_COLLECTION_NAME = "RSPY_643_TEST_COLLECTION"
collection = create_test_collection(TEST_COLLECTION_NAME)

# Check the catalog for RSPY_643_TEST_COLLECTION
items = catalog_client.get_items(TEST_COLLECTION_NAME)
assert not list(items)

#CADIP_SESSION_FILTER = "id=S3A_20250109134406046340" # Session id "platform='sentinel-1a'" "id=S1A_20200105072204051312" S3A_20250109134406046340 | S1A_20200105072204051312
CADIP_SESSION_FILTER ="id=S1A_20200105072204051312"


14:14:01.387 [INFO] (rs_client.rs_client) Retrieving all items from collection 'jgaucher:RSPY_643_TEST_COLLECTION'.


In [ ]:
# Other imports
import getpass
import os
import os.path as osp
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

flow_parameters = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_l0_demo_payload_dpr_mockup_template.yaml",
    "output_data_dir": f"{s3_output}/s3",
    "owner_id": OWNER_ID,
    "collection_name": TEST_COLLECTION_NAME,
    "cadip_stac_filter": CADIP_SESSION_FILTER,
    "staging_timeout": 120,
    "use_dpr_mockup": USE_DPR_MOCKUP,
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

14:14:01.541 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'rs-cluster-temp' path 'sub/dir/users/jgaucher/l0/config/logging_config.yaml'.

14:14:01.548 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml' to the bucket 'rs-cluster-temp' path 'sub/dir/users/jgaucher/l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml'.

14:14:01.550 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_3A.yaml' to the bucket 'rs-cluster-temp' path 'sub/dir/users/jgaucher/l0/config/s3/l0_processor_configuration_3A.yaml'.

14:14:01.552 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_dpr_mockup.yaml' to the bucket 'rs-cluster-temp' path 'sub/dir/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'.

14:14:01.621 | INFO    | prefect.S3Bucket - Uploaded 4 files from 'l0/config' to the bucket 'rs-cluster-temp' path 'sub/dir/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'

In [7]:
# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name
if cluster_mode:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_ADDRESS"]

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await share_bucket.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./s3l0_demo_processor_with_dpr_service.yaml"

In [ ]:
deploy_name = "s3l0-demo-processor/sprint23-s3l0-demo-processor"
await prefect_utils.wait_for_deployment(deploy_name)

## 3. Run Prefect flow

In [ ]:
output_data_dir = flow_parameters["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters) # flow parameters

In [ ]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)
eopf_prod_ids = ["S03MWRL0__20221101T092439_6037_A307_T677", "S03OLCL0__20210629T044945_0119_A247_T219"]
for id in eopf_prod_ids:
    assert catalog_client.get_item(TEST_COLLECTION_NAME, id) 
   

## 6. Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)
    dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [8]:
from importlib import reload
debug_flow = True

In [ ]:
if debug_flow:
    # shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_eopf(scale=2)
    # init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *
    os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
    os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name

In [30]:
if debug_flow:
    import s3l0_demo_processor_with_dpr_service
    reload(s3l0_demo_processor_with_dpr_service)
    init_demo()
    results = s3l0_demo_processor_with_dpr_service.s3l0_demo_processor(**flow_parameters)
    display(results)

/home/jgaucher/projects/rspy/working/demo-venv/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+----------+
| Package     | Client   | Scheduler | Workers  |
+-------------+----------+-----------+----------+
| dask        | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| distributed | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| lz4         | 4.4.3    | 4.4.4     | 4.4.4    |
| numpy       | 1.26.4   | 2.2.4     | 2.2.4    |
+-------------+----------+-----------+----------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Auxip service: http://localhost:8001/auxip
CADIP service: http://localhost:8002/cadip
Catalog service: http://localhost:8003
Staging service: http://localhost:8004


17:58:33.414 | INFO    | Flow run 'orange-boobook' - Beginning flow run 'orange-boobook' for flow 's3l0-demo-processor'

17:58:33.419 | INFO    | Flow run 'orange-boobook' - View at http://localhost:4200/runs/flow-run/76cab491-2d36-42b8-81a2-06a1fbb53625

17:58:33.691 | INFO    | Flow run 'orange-boobook' - For s3_l0_processor found module: l0.s3.s3_l0_processor and processing_unit: S3L0Processor

...

18:01:26.810 | ERROR   | Task run 'publish-to-catalog-7e9' - Exception in publishing to catalog: string indices must be integers, not 'str'

18:01:26.822 | INFO    | Task run 'publish-to-catalog-7e9' - Finished in state Completed()

18:01:26.836 | ERROR   | Flow run 'orange-boobook' - Encountered exception during execution: RuntimeError('Failed to publish to catalog')
Traceback (most recent call last):
  File "/home/jgaucher/projects/rspy/working/demo-venv/lib/python3.11/site-packages/prefect/flow_engine.py", line 765, in run_context
    yield self
  File "/home/jgaucher/projects/rspy/working/demo-venv/lib/python3.11/site-packages/prefect/flow_engine.py", line 1373, in run_flow_sync
    engine.call_flow_fn()
  File "/home/jgaucher/projects/rspy/working/demo-venv/lib/python3.11/site-packages/prefect/flow_engine.py", line 785, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jgaucher/projects/rspy/working/demo-venv/lib/python3.11/site-packages/prefect/utilities/callables.py", line 208, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/home/jgaucher/projects/rspy/github/rs-demo/notebooks/sprints/sprint23/s3l0_demo_processor_with_dpr_service.py", line 266, in s3l0_demo_processor
    raise RuntimeError("Failed to publish to catalog")
RuntimeError: Failed to publish to catalog

18:01:26.891 | ERROR   | Flow run 'orange-boobook' - Finished in state Failed('Flow run encountered an exception: RuntimeError: Failed to publish to catalog')

RuntimeError: Failed to publish to catalog